# Aquaculture Model Analysis

This notebook demonstrates how to analyze a trained model from the aquaculture competition framework using actual competition data.

In [ ]:
import numpy as np
import pandas as pd
import os
import random
from pathlib import Path
import sys
import re
import joblib
import yaml
import matplotlib.pyplot as plt
import optuna
import optuna.visualization as vis
# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
# Import our custom modules
from aquaculture.feature_engineering import AquacultureFeatureEngineer
from aquaculture.feature_selection import FeatureSelector  # For feature selection capabilities
from aquaculture.config import AquacultureConfig
from src.inference import load_inference_pipeline
from src.plotting import (
    plot_feature_importance, plot_roc_curve, plot_precision_recall_curve,
    plot_confusion_matrix, plot_calibration_curve
)
from src.metrics import calculate_metrics, competition_score, calculate_roc_curve, calculate_precision_recall_curve
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve
import shap
# For reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
# Set up paths
DATA_DIR = Path('../data')
EXPERIMENTS_DIR = Path('../experiments')

# Try to find the most recent experiment directory
if EXPERIMENTS_DIR.exists():
    experiment_dirs = [d for d in EXPERIMENTS_DIR.iterdir() if d.is_dir()]
    if experiment_dirs:
        # Sort by modification time (newest first)
        experiment_dirs.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_experiment = experiment_dirs[0]
        print(f"Found experiment: {latest_experiment.name}")
    else:
        print("No experiment directories found!")
        sys.exit(1)
else:
    print("Experiments directory not found!")
    sys.exit(1)

# Try to load the trainer first (which includes feature engineer and selector)
trainer_path = latest_experiment / "trainer.pkl"
trainer = None
if trainer_path.exists():
    try:
        import pickle
        with open(trainer_path, 'rb') as f:
            trainer = pickle.load(f)
        print("Trainer loaded successfully (includes feature engineer and selector)")
    except Exception as e:
        print(f"Warning: Could not load trainer.pkl: {e}")
        trainer = None

# Load the inference pipeline as fallback
print("Loading inference pipeline...")
try:
    pipeline = load_inference_pipeline(str(latest_experiment))
    print("✓ Inference pipeline loaded successfully")
    print(f"Model type: {type(pipeline.model).__name__}")
    if pipeline.feature_names:
        print(f"Number of features: {len(pipeline.feature_names)}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please check that the experiment directory exists and contains a trained model")
    sys.exit(1)

In [ ]:
# Load training data from CSV file
print("Loading training data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"Training data columns: {list(train_df.columns)}")

# Prepare data for training
print("Preparing data for training...")
# The target column is 'label' in the training data
# Feature columns are all columns except ID and label
feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
X_flat = train_df[feature_cols].values  # raw feature matrix (flattened)
# Reshape to 3D as expected by the feature engineer: (n_samples, 12, 12)
n_samples = X_flat.shape[0]
X = X_flat.reshape(n_samples, 12, 12)
# Get target variable - binary classification: 0 (no pond) or 1 (pond)
print("Extracting target variable from 'label' column...")
y = train_df['label'].values
print(f"Target variable shape: {y.shape}")
print(f"Target distribution: {np.bincount(y.astype(int)) if len(y) > 0 else 'empty'}")


In [ ]:
# Load experiment config to get feature engineering settings
config_path = latest_experiment / "config.yaml"
with open(config_path, 'r') as f:
    # Use FullLoader to allow construction of Python tuples (e.g., window_length_probs)
    config_dict = yaml.load(f, Loader=yaml.FullLoader)
# The TrainingConfig stores feature_engineering_config as a dict
feat_cfg_dict = config_dict.get('feature_engineering_config', {})
# If it's empty, we can also try to load via TrainingConfig (but it may not have the attr)
feature_engineering_config = AquacultureConfig(**feat_cfg_dict) if feat_cfg_dict else AquacultureConfig()
print(f"Loaded feature engineering config: {feature_engineering_config}")


In [ ]:
# Try to get the already‑fitted feature engineer from the trainer
trainer_path = latest_experiment / "trainer.pkl"
feature_engineer = None
if trainer_path.exists():
    try:
        with open(trainer_path, 'rb') as f:
            trainer_obj = pickle.load(f)
        # The trainer may have a feature_engineer attribute
        if hasattr(trainer_obj, 'feature_engineer') and trainer_obj.feature_engineer is not None:
            feature_engineer = trainer_obj.feature_engineer
            print("Retrieved fitted feature engineer from trainer.pkl")
        else:
            print("Trainer loaded but no feature_engineer attribute found.")
    except Exception as e:
        print(f"Failed to load trainer.pkl: {e}")
else:
    print("trainer.pkl not found.")

# If we don't have a fitted engineer, create a fresh one and fit it
if feature_engineer is None:
    feature_engineer = AquacultureFeatureEngineer(
        simulate_mask=feature_engineering_config.simulate_mask,
        random_state=feature_engineering_config.random_state,
        window_length_probs=feature_engineering_config.window_length_probs,
        start_month_distribution=feature_engineering_config.start_month_distribution,
        s2_monthly_dropout=feature_engineering_config.s2_monthly_dropout,
        include_optical=feature_engineering_config.include_optical,
        include_sar=feature_engineering_config.include_sar,
        include_cross_sensor_features=feature_engineering_config.include_cross_sensor_features,
        include_temporal_statistics=feature_engineering_config.include_temporal_statistics,
        include_normalized_optical=feature_engineering_config.include_normalized_optical,
        include_directional_vote=feature_engineering_config.include_directional_vote,
        include_conditional_features=feature_engineering_config.include_conditional_features,
        include_metadata=feature_engineering_config.include_metadata
    )
    print("Created new AquacultureFeatureEngineer instance.")
    # Fit on raw data (just to set internal shapes/feature names)
    feature_engineer.fit(X)
    print("Fitted feature engineer on raw data.")


In [ ]:
# Transform raw data using the feature engineer with training=True
# This applies stochastic window selection and S2‑band dropout.
X_features = feature_engineer.transform(X, training=True)
X_features = X_features.values  # ensure numpy array
print(f"Reconstructed feature matrix shape: {X_features.shape}")
# Optional: show first few feature names
if hasattr(feature_engineer, 'feature_names_out_'):
    print(f"First 5 feature names: {list(feature_engineer.feature_names_out_)[:5]}")
elif hasattr(feature_engineer, 'get_feature_names_out'):
    try:
        names = feature_engineer.get_feature_names_out()
        print(f"First 5 feature names: {list(names)[:5]}")
    except Exception:
        pass


In [ ]:
# Make predictions using the reconstructed features
print("Generating predictions...")
if trainer is not None:
    # Use the trainer's predict methods which handle feature engineering and selection
    predictions = trainer.predict(X, training=True)
    probabilities = trainer.predict_proba(X, training=True)[:, 1]
    print("✓ Predictions generated using trainer (includes feature engineering and selection)")
else:
    # Fallback: use the pipeline directly on pre-computed features
    # Use the model directly to avoid double feature transformation
    predictions = pipeline.model.predict(X_features)
    probabilities = pipeline.model.predict_proba(X_features)[:, 1]
    # Note: predict_proba returns shape (n_samples, 2); we take column 1 for positive class
    print("✓ Predictions generated using pipeline (fallback method)")

## 5. Evaluate Model Performance


In [ ]:
# Calculate metrics for single target
print(f"\n=== Target Evaluation ===")
target_pred = predictions
target_prob = probabilities
target_true = y

# Calculate various metrics
metrics = calculate_metrics(target_true, target_prob)

# Print key metrics
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")
print(f"ROC AUC:   {metrics['roc_auc']:.4f}")
print(f"PR AUC:    {metrics['pr_auc']:.4f}")

# Calculate competition score (for single target, this is just the standard competition score)
competition_score_value = competition_score(target_true, target_prob)
print(f"\nCompetition Score: {competition_score_value:.4f}")

## 6. Generate Visualizations


In [ ]:
# Generate plots for single target
print(f"\nGenerating plots for target...")
target_pred = predictions
target_prob = probabilities
target_true = y

# Create a directory for plots
PLOTS_DIR = EXPERIMENTS_DIR / latest_experiment.name / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

# ROC Curve
fpr, tpr, _ = calculate_roc_curve(target_true, target_prob)
roc_auc = metrics['roc_auc']
plot_roc_curve(fpr, tpr, roc_auc,
               title=f'ROC Curve',
               save_path=PLOTS_DIR / f'roc_curve.png')

# Precision-Recall Curve
precision, recall, _ = calculate_precision_recall_curve(target_true, target_prob)
pr_auc = metrics['pr_auc']
plot_precision_recall_curve(precision, recall, pr_auc,
                            title=f'Precision-Recall Curve',
                            save_path=PLOTS_DIR / f'pr_curve.png')

# Confusion Matrix
cm = confusion_matrix(target_true, target_pred)
plot_confusion_matrix(cm,
                      title=f'Confusion Matrix',
                      save_path=PLOTS_DIR / f'confusion_matrix.png')

# Calibration Curve
prob_true, prob_pred = calibration_curve(target_true, target_prob, n_bins=10)
plot_calibration_curve(prob_true, prob_pred,
                       title=f'Calibration Curve',
                       save_path=PLOTS_DIR / f'calibration_curve.png')

print("\nAll plots generated successfully!")

## 7. SHAP Feature Importance Analysis

If SHAP values were computed during training (compute_shap: true), they are saved in the experiment directory. This section loads and visualizes them.


In [ ]:
# Load SHAP artifacts if they exist

exp_dir = latest_experiment  # from earlier
shap_path = exp_dir / "shap_values.npz"
config_path = exp_dir / "config.yaml"

if shap_path.exists():
    print(f"Loading SHAP values from {shap_path}")
    shap_data = np.load(shap_path)
    shap_values = shap_data["shap_values"]
    feature_names = shap_data["feature_names"].tolist()
    print(f"SHAP values shape: {shap_values.shape}")
    print(f"Number of features: {len(feature_names)}")
    
    # Load config to get sampling parameters
    with open(config_path, 'r') as f:
        config_dict = yaml.load(f, Loader=yaml.FullLoader)
    # The config is stored as a dict; we need to get the TrainingConfig fields
    # For simplicity, we extract what we need
    random_seed = config_dict.get('random_seed', 42)
    shap_sample_size = config_dict.get('shap_sample_size', 100)
    
    # CRITICAL: We must use the same feature transformation that was used during SHAP computation
    # The SHAP values were computed using the trainer's pipeline (feature engineering + selection)
    if trainer is not None:
        # Get features exactly as they were used during training (for SHAP consistency)
        # This applies both feature engineering and feature selection
        X_features_transformed = trainer._prepare_data(X, np.zeros(X.shape[0]), training=True)[0]
        n_samples = X_features_transformed.shape[0]
        print(f"Features after full transform (eng + select): {X_features_transformed.shape}")
        print(f"Expected features from SHAP: {shap_values.shape[1]}")
        
        # Verify we have the right number of features
        if X_features_transformed.shape[1] == shap_values.shape[1]:
            print("✓ Feature count matches between transformed data and SHAP values")
        else:
            print(f"⚠ Feature count mismatch: got {X_features_transformed.shape[1]}, expected {shap_values.shape[1]}")
            # Try to get feature names from the trainer if available
            if hasattr(trainer, 'feature_names') and trainer.feature_names is not None:
                print(f"Trainer feature names count: {len(trainer.feature_names)}")
                if trainer.feature_names is not None:
                    print(f"Trainer feature names count: {len(trainer.feature_names)}")
                    if len(trainer.feature_names) == shap_values.shape[1]:
                        feature_names = trainer.feature_names
                        print("✓ Using trainer feature names")
    else:
        # Fallback if no trainer available
        X_features_transformed = feature_engineer.transform(X, training=True).values
        n_samples = X_features_transformed.shape[0]
        print(f"Features after feature engineering only: {X_features_transformed.shape}")
    
    # Determine if sampling was applied during SHAP computation
    if shap_values.shape[0] == n_samples:
        # SHAP values were computed on the full dataset
        X_shap = X_features_transformed
        print("SHAP values computed on full dataset. Using all features for plotting.")
    else:
        # SHAP values were computed on a subset; reproduce the same sample
        sample_size = min(shap_sample_size, n_samples)
        if sample_size < n_samples:
            # Use the same random seed for reproducibility
            rng = np.random.RandomState(random_seed)
            indices = rng.choice(n_samples, size=sample_size, replace=False)
            X_shap = X_features_transformed[indices]
            print(f"SHAP values computed on a sample of {sample_size} rows. "
                  f"Using the same sampled features for plotting.")
        else:
            # sample_size >= n_samples, effectively full dataset
            X_shap = X_features_transformed
            print("Sample size >= number of samples. Using full dataset for plotting.")
    
    # Final verification
    if shap_values.shape[1] == X_shap.shape[1]:
        print(f"✓ Final shapes match: SHAP {shap_values.shape} vs X_shap {X_shap.shape}")
        
        # Compute mean absolute SHAP values (feature importance)
        shap_importance = np.mean(np.abs(shap_values), axis=0)
        # Create DataFrame for easy sorting
        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": shap_importance
        }).sort_values("importance", ascending=False)
        display(importance_df.head(20))
        # Optionally save to CSV
        importance_df.to_csv(exp_dir / "shap_feature_importance_from_notebook.csv", index=False)
    else:
        print(f"✗ Final shape mismatch: SHAP {shap_values.shape} vs X_shap {X_shap.shape}")
        print("Cannot proceed with plotting due to feature dimension mismatch.")
else:
    print("SHAP values not found (shap_values.npz).")
    print("Please ensure that in the training notebook (01_train_model.ipynb) you have:")
    print("  config.compute_shap = True")
    print("and re-run the training to generate the SHAP values.")
    print("Alternatively, you can view the pre-generated SHAP importance CSV and plots in:")
    print(f"  {exp_dir}/features/shap_feature_importance.csv")
    print(f"  {exp_dir}/plots/shap/")

### SHAP Visualizations

Generate summary and dependence plots.

In [ ]:
if 'shap_values' in locals() and 'X_shap' in locals() and 'feature_names' in locals():
    # Additional safety check: verify dimensions match before plotting
    if shap_values.shape[1] == X_shap.shape[1] == len(feature_names):
        # Use the experiment directory from earlier
        exp_dir = latest_experiment
        # Summary plot
        plt.figure()
        shap.summary_plot(shap_values, features=X_shap, feature_names=feature_names, show=False)
        plt.title('SHAP Summary Plot')
        plt.tight_layout()
        plot_path = exp_dir / 'shap_summary_notebook.png'
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved summary plot to {plot_path}')
    else:
        print(f"Dimension mismatch for plotting:")
        print(f"  SHAP values shape: {shap_values.shape}")
        print(f"  X_shap shape: {X_shap.shape}")
        print(f"  Feature names length: {len(feature_names)}")
        print("Skipping SHAP plot due to dimension mismatch.")
else:
    print('SHAP data not loaded completely. Please run the SHAP loading cell first.')

In [ ]:
if 'shap_values' in locals() and 'feature_names' in locals() and 'X_shap' in locals():
    exp_dir = latest_experiment
    # Determine top features by mean |SHAP|
    mean_abs = np.mean(np.abs(shap_values), axis=0)
    top_n = min(10, len(feature_names))
    top_indices = np.argsort(mean_abs)[::-1][:top_n]
    top_features = [feature_names[i] for i in top_indices]
    
    for feat in top_features:
        idx = feature_names.index(feat)
        # Additional safety check for each plot
        if shap_values.shape[1] == X_shap.shape[1]:
            plt.figure()
            shap.dependence_plot(idx, shap_values, features=X_shap, feature_names=feature_names, show=False)
            plt.title(f'SHAP Dependence: {feat}')
            plt.tight_layout()
            plot_path = exp_dir / f'shap_dependence_{feat.replace(" ", "_").replace("/", "_")}.png'
            plt.savefig(plot_path, dpi=150, bbox_inches='tight')
            plt.show()
            print(f'Saved dependence plot for {feat} to {plot_path}')
        else:
            print(f"Skipping dependence plot for {feat} due to dimension mismatch:")
            print(f"  SHAP values shape: {shap_values.shape}")
            print(f"  X_shap shape: {X_shap.shape}")
else:
    print('SHAP data not loaded completely. Please run the SHAP loading cell first.')

## 8. Optuna Study Exploration

This notebook demonstrates how to analyze a trained model from the aquaculture competition framework using actual competition data.


In [ ]:


experiments_root = Path("../experiments")
# Find directories matching timestamp pattern
exp_folders = sorted(
    [p for p in experiments_root.iterdir() if p.is_dir() and re.match(r"\d{8}_\d{6}", p.name)],
    key=lambda p: p.name,
    reverse=True
)
if not exp_folders:
    raise FileNotFoundError("No experiment folders found in ../experiments")
latest_exp = exp_folders[0]
print(f"Using experiment folder: {latest_exp}")

study_path = latest_exp / "models" / "optuna_study.pkl"
study = joblib.load(study_path)
print(f"Loaded Optuna study with {len(study.trials)} trials.")


### 8.1 Study Overview


In [ ]:

best = study.best_trial
print(f"Best trial number: {best.number}")
print(f"Best value (competition score): {best.value:.5f}")
print("Best hyperparameters:")
for k, v in best.params.items():
    print(f"  {k}: {v}")


### 8.2 Trials DataFrame


In [ ]:

trials_df = study.trials_dataframe()
# Select a subset of columns for readability
cols = ["number", "value", "params_model_type", "params_learning_rate",
        "params_n_estimators", "params_max_depth", "params_subsample",
        "params_colsample_bytree", "state"]
# Keep only those that exist
existing_cols = [c for c in cols if c in trials_df.columns]
print(trials_df[existing_cols].head(10))
print(f"Total trials: {len(trials_df)}")


### 8.3 Hyperparameter Importance


In [ ]:

try:
    impt = optuna.importance.get_param_importances(study)
    if impt:
        # Sort
        sorted_impt = sorted(impt.items(), key=lambda x: x[1], reverse=True)
        params, importances = zip(*sorted_impt)
        plt.figure(figsize=(8,6))
        plt.barh(params, importances)
        plt.xlabel("Importance")
        plt.title("Hyperparameter Importance")
        plt.gca().invert_yaxis()  # highest on top
        plt.tight_layout()
        plt.show()
    else:
        print("Could not compute parameter importance (no trials with sufficient info).")
except Exception as e:
    print(f"Error plotting parameter importance: {e}")


### 8.4 Optimization History


In [ ]:

try:
    fig = vis.plot_optimization_history(study)
    fig.show()
except Exception as e:
    print(f"Error plotting optimization history: {e}")


### 8.5 Parallel Coordinate Plot


In [ ]:

try:
    fig = vis.plot_parallel_coordinate(study)
    fig.show()
except Exception as e:
    print(f"Error plotting parallel coordinate: {e}")


### 8.6 Slice Plot


In [ ]:

try:
    fig = vis.plot_slice(study)
    fig.show()
except Exception as e:
    print(f"Error plotting slice: {e}")


### 8.7 CV Score Evolution with Uncertainty Bands

Plot how the mean cross-validation score and its uncertainty evolve 
over the course of optimization. This shows both the progress of 
optimization and the stability of the parameter settings.


In [ ]:
try:
    # Extract trials that completed successfully
    trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if trials:
        trial_numbers = [t.number for t in trials]
        cv_means = [t.user_attrs.get("cv_mean_score", 0) for t in trials]
        cv_stds = [t.user_attrs.get("cv_std_score", 0) for t in trials]
        
        # Create the plot
        plt.figure(figsize=(12, 6))
        
        # Plot mean CV score with error bands
        plt.plot(trial_numbers, cv_means, 'b-o', linewidth=2, markersize=4, label='Mean CV Score')
        plt.fill_between(trial_numbers, 
                         np.array(cv_means) - np.array(cv_stds),
                         np.array(cv_means) + np.array(cv_stds),
                         alpha=0.3, color='blue', label='±1 Standard Deviation')
        
        # Mark the best trial
        best_trial = study.best_trial
        plt.axvline(x=best_trial.number, color='red', linestyle='--', alpha=0.7, 
                   label=f'Best Trial ({best_trial.number})')
        plt.axhline(y=best_trial.value, color='red', linestyle=':', alpha=0.7,
                   label=f'Best Score ({best_trial.value:.4f})')
        
        plt.xlabel('Trial Number')
        plt.ylabel('Competition Score')
        plt.title('Cross-Validation Score Evolution During Optimization\n(Shaded area shows ±1 std dev across folds)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Print some summary statistics
        print(f"Best CV score: {best_trial.value:.4f} (trial {best_trial.number})")
        print(f"Final CV score: {cv_means[-1]:.4f} ± {cv_stds[-1]:.4f} (trial {trial_numbers[-1]})")
        print(f"Improvement: {cv_means[-1] - cv_means[0]:.4f} over {len(trials)} trials")
    else:
        print("No completed trials found for visualization.")
except Exception as e:
    print(f"Error plotting CV evolution: {e}")

### 8.8 Fold Score Distribution Across Trials

Box plot showing the distribution of scores across CV folds for each trial.
This helps understand the variance and consistency of performance 
across different data splits during optimization.


In [ ]:
try:
    # Extract trials that completed successfully
    trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if trials:
        # Determine number of folds from the first trial (assuming consistent CV setup)
        sample_trial = trials[0]
        fold_keys = [k for k in sample_trial.user_attrs.keys() if k.startswith('fold_') and k.endswith('_score')]
        n_folds = len(fold_keys)
        
        if n_folds > 0:
            # Prepare data for box plot: each box represents one fold across all trials
            fold_data = []
            fold_labels = []
            
            for i in range(n_folds):
                fold_scores = [t.user_attrs.get(f'fold_{i}_score', 0) for t in trials]
                fold_data.append(fold_scores)
                fold_labels.append(f'Fold {i+1}')
            
            # Create the plot
            plt.figure(figsize=(12, 6))
            
            box_plot = plt.boxplot(fold_data, labels=fold_labels, patch_artist=True)
            
            # Customize box plot appearance
            for patch in box_plot['boxes']:
                patch.set_facecolor('lightblue')
                patch.set_alpha(0.7)
            
            # Add mean points for each fold
            fold_means = [np.mean(fold_data[i]) for i in range(n_folds)]
            plt.scatter(range(1, n_folds+1), fold_means, 
                       color='red', zorder=5, s=50, label='Mean per fold', 
                       edgecolors='darkred', linewidth=1)
            
            # Add overall statistics
            all_scores = [score for sublist in fold_data for score in sublist]
            overall_mean = np.mean(all_scores)
            overall_std = np.std(all_scores)
            
            plt.axhline(y=overall_mean, color='green', linestyle='-', alpha=0.7,
                       label=f'Overall Mean ({overall_mean:.4f})')
            plt.axhline(y=overall_mean + overall_std, color='green', linestyle='--', alpha=0.5,
                       label=f'+1 Std Dev ({overall_std:.4f})')
            plt.axhline(y=overall_mean - overall_std, color='green', linestyle='--', alpha=0.5,
                       label=f'-1 Std Dev ({overall_std:.4f})')
            
            plt.xlabel('CV Fold')
            plt.ylabel('Competition Score')
            plt.title(f'Distribution of Fold Scores Across {len(trials)} Optimization Trials\n'
                     f'(Shows consistency of performance across different data splits)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            
            # Print summary statistics
            print(f"Overall statistics across all folds and trials:")
            print(f"  Mean: {overall_mean:.4f}")
            print(f"  Std:  {overall_std:.4f}")
            print(f"  Min:  {np.min(all_scores):.4f}")
            print(f"  Max:  {np.max(all_scores):.4f}")
        else:
            print("No fold score data found in trials.")
    else:
        print("No completed trials found for visualization.")
except Exception as e:
    print(f"Error plotting fold distribution: {e}")

### 8.9 Optimization Progress Summary

Summary statistics showing the progress of optimization over time.
